# 📘 Session 1: Supervised Machine Learning — Interactive Demo

This notebook provides **interactive demonstrations** for all the concepts from Session 1:

1. **Linear Regression Model** — Visualize $y = b + wx$ for different parameter values
2. **Loss Function** — See how MSE changes with different $w$ and $b$
3. **Gradient Descent** — Watch the optimization process step-by-step
4. **Full Implementation** — Manual gradient descent vs sklearn, plus analytical solution

In [ ]:
# ── Setup: Install & Import ──────────────────────────────────────────
import subprocess, sys
for pkg in ['matplotlib', 'numpy', 'scikit-learn', 'ipywidgets']:
    try:
        __import__(pkg.replace('-', '_').replace('scikit_learn', 'sklearn'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interact, FloatSlider, IntSlider
from IPython.display import display, HTML, Markdown, clear_output
from matplotlib.gridspec import GridSpec

%matplotlib inline

# ── Dark theme ──
plt.style.use('dark_background')
matplotlib.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor': '#16213e',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#a0a0a0',
    'ytick.color': '#a0a0a0',
    'axes.edgecolor': '#333366',
    'grid.color': '#2a2a4a',
    'grid.alpha': 0.4,
    'axes.grid': True,
})

# Color palette
C = {
    'accent':    '#e94560',
    'cyan':      '#00d2ff',
    'teal':      '#0cebeb',
    'purple':    '#533483',
    'yellow':    '#f9d423',
    'orange':    '#ff6b35',
    'green':     '#2ecc71',
}

print('✅ All imports successful! Ready to go.')

---
## 1️⃣ The Linear Regression Model: $y = b + wx$

A linear model is defined by just **two parameters**:
- $b$ (bias / intercept) — where the line crosses the y-axis
- $w$ (weight / slope) — how steep the line is

### 1a. Static Comparison — Different Values of $w$ and $b$

In [ ]:
def plot_interactive_line(w=1.0, b=0.0):
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Generate x values
    x = np.linspace(-10, 10, 200)
    y = b + w * x
    
    # Plot the line (keeping your cyan color preference)
    ax.plot(x, y, color='c', lw=2.5, label=f'y = {b:.1f} + {w:.1f}x')
    
    # --- Move main axes to (0, 0) ---
    ax.spines['left'].set_position('zero')
    ax.spines['bottom'].set_position('zero')
    
    # Hide the top and right spines completely
    ax.spines['right'].set_color('none')
    ax.spines['top'].set_color('none')
    
    # Ensure ticks show up correctly on the centered axes
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')
    
    # Lock the axis limits! 
    # This prevents the graph from violently zooming/jumping as you move the slider.
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    
    # Add a grid and legend
    ax.grid(True, linestyle=':', alpha=0.6)
    ax.legend(loc='upper left', fontsize=10)
    ax.set_title('Interactive: y = b + wx', fontweight='bold', pad=20)
    
    plt.show()

# Generate the interactive sliders
interact(plot_interactive_line, 
         w=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='Weight (w):'), 
         b=FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.5, description='Bias (b):'))

### 1b. 🎛️ Interactive — Adjust $w$ and $b$ yourself

Drag the sliders to change the parameters. The line updates with the exact formula as its label.

In [ ]:
# ── 1b: Interactive y = b + wx ──────────────────────────────────────
@interact(
    w=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description='w (slope):',
                  style={'description_width': '80px'}, readout_format='.1f'),
    b=FloatSlider(value=0.0, min=-10.0, max=10.0, step=0.1, description='b (bias):',
                  style={'description_width': '80px'}, readout_format='.1f')
)
def plot_line(w=1.0, b=0.0):
    x = np.linspace(-5, 5, 200)
    y = b + w * x

    fig, ax = plt.subplots(figsize=(9, 5))

    # Line + shading
    ax.plot(x, y, color=C['accent'], lw=3, zorder=5,
            label=f'y = {b:.1f} + {w:.1f}x')
    ax.fill_between(x, y, alpha=0.06, color=C['accent'])

    # Y-intercept marker
    ax.scatter([0], [b], color=C['yellow'], s=120, zorder=10,
               edgecolors='white', linewidth=2)
    ax.annotate(f'  b = {b:.1f}', (0, b), fontsize=11,
                color=C['yellow'], fontweight='bold')

    # Slope triangle
    if abs(w) > 0.05:
        x0 = 1.0
        y0 = b + w * x0
        ax.plot([x0, x0 + 1], [y0, y0], '--', color=C['teal'], lw=1.5, alpha=0.8)
        ax.plot([x0 + 1, x0 + 1], [y0, y0 + w], '--', color=C['teal'], lw=1.5, alpha=0.8)
        ax.annotate(f'Δy={w:.1f}', (x0 + 1.1, y0 + w / 2), fontsize=9, color=C['teal'])
        ax.annotate(f'Δx=1', (x0 + 0.3, y0 - 0.5), fontsize=9, color=C['teal'])

    ax.axhline(0, color='#555', lw=0.8); ax.axvline(0, color='#555', lw=0.8)
    ax.set_xlim(-5, 5); ax.set_ylim(-15, 15)
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_title(f'y = {b:.1f} + {w:.1f}x', fontsize=16,
                 fontweight='bold', color=C['cyan'])
    ax.legend(fontsize=11, loc='upper left', facecolor='#1a1a2e', edgecolor='#444')
    plt.tight_layout()
    plt.show()

---
## 2️⃣ Loss Function: $L[\phi] = \sum_{i=1}^{I}(b + wx_i - y_i)^2$

The loss tells us how **wrong** our model's predictions are.
We square each error so that:
- Positive and negative errors don't cancel out
- Larger errors are penalized more heavily

### 2a. Static — Loss Surface Over $w$ and $b$

In [ ]:
# ── 2a: 3D Loss surface ─────────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

# Generate toy data with known true parameters
np.random.seed(42)
N_PTS = 20
x_data = np.random.uniform(-3, 3, N_PTS)
TRUE_W, TRUE_B = 2.0, 1.0
y_data = TRUE_B + TRUE_W * x_data + np.random.normal(0, 1.0, N_PTS)

# Compute loss grid
w_range = np.linspace(-2, 6, 100)
b_range = np.linspace(-4, 6, 100)
W, B = np.meshgrid(w_range, b_range)
Loss = np.zeros_like(W)
for i in range(N_PTS):
    Loss += (B + W * x_data[i] - y_data[i]) ** 2
Loss /= N_PTS

fig = plt.figure(figsize=(14, 5))

# --- Left: 3D surface ---
ax1 = fig.add_subplot(121, projection='3d')
ax1.plot_surface(W, B, Loss, cmap='magma', alpha=0.85, edgecolor='none')
ax1.scatter([TRUE_W], [TRUE_B], [0], color=C['teal'], s=100, zorder=10)
ax1.set_xlabel('w'); ax1.set_ylabel('b'); ax1.set_zlabel('MSE')
ax1.set_title('Loss Surface (3D)', color=C['cyan'], fontweight='bold')
ax1.view_init(elev=30, azim=-60)
ax1.set_facecolor('#16213e')

# --- Right: Contour ---
ax2 = fig.add_subplot(122)
cp = ax2.contourf(W, B, Loss, levels=40, cmap='magma')
ax2.contour(W, B, Loss, levels=15, colors='white', linewidths=0.3, alpha=0.4)
plt.colorbar(cp, ax=ax2, label='MSE Loss')
ax2.scatter([TRUE_W], [TRUE_B], color=C['teal'], s=120,
            edgecolors='white', linewidth=2, zorder=10,
            label=f'True (w={TRUE_W}, b={TRUE_B})')
ax2.set_xlabel('w'); ax2.set_ylabel('b')
ax2.set_title('Loss Contour (Top View)', color=C['cyan'], fontweight='bold')
ax2.legend(facecolor='#1a1a2e', edgecolor='#444')

plt.tight_layout()
plt.show()
print('\n💡 The bowl shape = single global minimum — the best (w, b).')
print(f'   True parameters: w = {TRUE_W}, b = {TRUE_B}')

### 2b. 🎛️ Interactive — Compute Loss for Your Own $w$ and $b$

Drag the sliders to pick $w$ and $b$, see the prediction line vs data points,
and watch how the loss changes. Red dashed lines = individual errors.

Use the **seed** slider to generate different random datasets.

In [ ]:
# ── 2b: Interactive loss visualization ──────────────────────────────
@interact(
    w=FloatSlider(value=1.0, min=-4.0, max=6.0, step=0.1, description='w:',
                  style={'description_width': '40px'}, readout_format='.1f'),
    b=FloatSlider(value=0.0, min=-6.0, max=6.0, step=0.1, description='b:',
                  style={'description_width': '40px'}, readout_format='.1f'),
    seed=IntSlider(value=42, min=0, max=100, step=1, description='Data Seed:',
                   style={'description_width': '80px'})
)
def plot_loss_interactive(w=1.0, b=0.0, seed=42):
    # Generate data for this seed
    rng = np.random.RandomState(seed)
    n = 20
    xd = rng.uniform(-3, 3, n)
    tw = rng.uniform(0.5, 3.0)
    tb = rng.uniform(-1.0, 2.0)
    yd = tb + tw * xd + rng.normal(0, 1.0, n)

    y_pred = b + w * xd
    errors = y_pred - yd
    mse = np.mean(errors ** 2)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: data + line + error bars ──
    ax = axes[0]
    x_line = np.linspace(xd.min() - 1, xd.max() + 1, 100)
    y_line = b + w * x_line
    ax.plot(x_line, y_line, color=C['accent'], lw=2.5, zorder=5,
            label=f'y = {b:.1f} + {w:.1f}x')
    ax.scatter(xd, yd, color=C['cyan'], s=60, zorder=8,
               edgecolors='white', linewidth=1, label='Data points')

    # Error lines
    for xi, yi, yp in zip(xd, yd, y_pred):
        ax.plot([xi, xi], [yi, yp], '--', color='#ff6b6b', lw=1, alpha=0.7)

    ax.set_title(f'Data vs Prediction   |   MSE = {mse:.2f}',
                 fontweight='bold', color=C['yellow'])
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(facecolor='#1a1a2e', edgecolor='#444')

    # ── Right: loss as function of w (b fixed) ──
    ax2 = axes[1]
    w_scan = np.linspace(-4, 6, 200)
    loss_scan = np.array([np.mean((b + wi * xd - yd) ** 2) for wi in w_scan])
    ax2.plot(w_scan, loss_scan, color=C['teal'], lw=2.5)
    ax2.fill_between(w_scan, loss_scan, alpha=0.1, color=C['teal'])
    ax2.scatter([w], [mse], color=C['accent'], s=150, zorder=10,
                edgecolors='white', linewidth=2)
    ax2.annotate(f'  w={w:.1f}\n  Loss={mse:.2f}', (w, mse),
                 fontsize=10, color=C['accent'], fontweight='bold')
    ax2.set_title(f'Loss vs w  (b fixed at {b:.1f})',
                  fontweight='bold', color=C['teal'])
    ax2.set_xlabel('w'); ax2.set_ylabel('MSE Loss')

    plt.tight_layout()
    plt.show()

---
## 3️⃣ Gradient Descent

Instead of trying every possible $(w, b)$, gradient descent **iteratively updates** the parameters
by moving in the direction of steepest decrease:

$$w \leftarrow w - \alpha \frac{\partial L}{\partial w} \qquad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

where:
$$\frac{\partial L}{\partial w} = \frac{2}{I}\sum_{i=1}^{I}(b + wx_i - y_i)\,x_i \qquad \frac{\partial L}{\partial b} = \frac{2}{I}\sum_{i=1}^{I}(b + wx_i - y_i)$$

### 3. 🎛️ Interactive Gradient Descent Visualization

Adjust the **learning rate**, **starting point**, and **number of steps** to see how gradient descent converges.

Try:
- A **very small** learning rate (0.001) → slow convergence
- A **very large** learning rate (0.5) → may overshoot / diverge!
- Different **starting points** → all converge to the same minimum

In [ ]:
# ── 3: Interactive Gradient Descent ─────────────────────────────────
# Uses the data from Section 2a (x_data, y_data with TRUE_W=2, TRUE_B=1)

@interact(
    lr=FloatSlider(value=0.01, min=0.0001, max=0.5, step=0.001,
                   description='α (lr):', readout_format='.3f',
                   style={'description_width': '70px'}),
    n_steps=IntSlider(value=50, min=5, max=300, step=5,
                      description='Steps:',
                      style={'description_width': '70px'}),
    w_init=FloatSlider(value=-2.0, min=-4.0, max=6.0, step=0.2,
                       description='w₀:', readout_format='.1f',
                       style={'description_width': '70px'}),
    b_init=FloatSlider(value=-3.0, min=-6.0, max=6.0, step=0.2,
                       description='b₀:', readout_format='.1f',
                       style={'description_width': '70px'})
)
def run_gd_interactive(lr=0.01, n_steps=50, w_init=-2.0, b_init=-3.0):
    w, b = w_init, b_init
    n = len(x_data)

    w_hist, b_hist = [w], [b]
    loss_hist = [np.mean((b + w * x_data - y_data) ** 2)]

    for _ in range(n_steps):
        errors = (b + w * x_data) - y_data
        dL_dw = (2 / n) * np.sum(errors * x_data)
        dL_db = (2 / n) * np.sum(errors)
        w -= lr * dL_dw
        b -= lr * dL_db
        w_hist.append(w)
        b_hist.append(b)
        loss_hist.append(np.mean((b + w * x_data - y_data) ** 2))

    w_hist = np.array(w_hist)
    b_hist = np.array(b_hist)
    loss_hist = np.array(loss_hist)

    # ── Build 4-panel figure ──
    fig = plt.figure(figsize=(16, 10))
    gs = GridSpec(2, 2, figure=fig, hspace=0.35, wspace=0.3)

    # ── Top-Left: Loss contour + GD path ──
    ax1 = fig.add_subplot(gs[0, 0])
    wr = np.linspace(-4, 6, 100)
    br = np.linspace(-6, 6, 100)
    Wg, Bg = np.meshgrid(wr, br)
    Lg = np.zeros_like(Wg)
    for i in range(n):
        Lg += (Bg + Wg * x_data[i] - y_data[i]) ** 2
    Lg /= n

    ax1.contourf(Wg, Bg, Lg, levels=40, cmap='magma', alpha=0.9)
    ax1.contour(Wg, Bg, Lg, levels=20, colors='white', linewidths=0.3, alpha=0.3)
    ax1.plot(w_hist, b_hist, 'o-', color=C['teal'], markersize=3,
             lw=1.5, alpha=0.9, label='GD Path')
    ax1.scatter(w_hist[0], b_hist[0], color=C['yellow'], s=120,
                edgecolors='white', linewidth=2, zorder=10, label='Start')
    ax1.scatter(w_hist[-1], b_hist[-1], color=C['teal'], s=120, marker='*',
                edgecolors='white', linewidth=2, zorder=10, label='End')
    ax1.set_xlabel('w'); ax1.set_ylabel('b')
    ax1.set_title('GD Path on Loss Contour', fontweight='bold', color=C['cyan'])
    ax1.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#444')

    # ── Top-Right: Loss over steps ──
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.plot(loss_hist, color=C['accent'], lw=2.5)
    ax2.fill_between(range(len(loss_hist)), loss_hist, alpha=0.15, color=C['accent'])
    ax2.scatter(0, loss_hist[0], color=C['yellow'], s=100,
                edgecolors='white', linewidth=2, zorder=10)
    ax2.scatter(len(loss_hist)-1, loss_hist[-1], color=C['teal'], s=100,
                marker='*', edgecolors='white', linewidth=2, zorder=10)
    ax2.set_xlabel('Step'); ax2.set_ylabel('MSE Loss')
    ax2.set_title('Loss Over Time', fontweight='bold', color=C['accent'])
    ax2.annotate(f'Final Loss: {loss_hist[-1]:.4f}',
                 xy=(len(loss_hist) * 0.5, loss_hist.max() * 0.7),
                 fontsize=12, color=C['teal'], fontweight='bold',
                 bbox=dict(facecolor='#1a1a2e', edgecolor=C['teal'],
                           boxstyle='round,pad=0.5', alpha=0.9))

    # ── Bottom-Left: w and b convergence ──
    ax3 = fig.add_subplot(gs[1, 0])
    ax3.plot(w_hist, color=C['cyan'], lw=2, label='w')
    ax3.plot(b_hist, color=C['yellow'], lw=2, label='b')
    ax3.axhline(TRUE_W, color=C['cyan'], ls='--', alpha=0.4, label=f'True w={TRUE_W}')
    ax3.axhline(TRUE_B, color=C['yellow'], ls='--', alpha=0.4, label=f'True b={TRUE_B}')
    ax3.set_xlabel('Step'); ax3.set_ylabel('Parameter Value')
    ax3.set_title('Parameter Convergence', fontweight='bold', color=C['yellow'])
    ax3.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#444')

    # ── Bottom-Right: Final fit ──
    ax4 = fig.add_subplot(gs[1, 1])
    xl = np.linspace(x_data.min() - 1, x_data.max() + 1, 100)
    ax4.scatter(x_data, y_data, color=C['cyan'], s=60, edgecolors='white',
                linewidth=1, zorder=8, label='Data')
    ax4.plot(xl, b_hist[-1] + w_hist[-1] * xl, color=C['accent'], lw=2.5,
             label=f'GD: y = {b_hist[-1]:.2f} + {w_hist[-1]:.2f}x')
    ax4.plot(xl, TRUE_B + TRUE_W * xl, '--', color=C['teal'], lw=1.5, alpha=0.7,
             label=f'True: y = {TRUE_B} + {TRUE_W}x')
    ax4.set_xlabel('x'); ax4.set_ylabel('y')
    ax4.set_title('Final Fit vs True', fontweight='bold', color=C['cyan'])
    ax4.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#444')

    fig.suptitle(f'Gradient Descent  |  α = {lr:.4f}  |  {n_steps} steps  |  '
                 f'Final w = {w_hist[-1]:.3f}, b = {b_hist[-1]:.3f}',
                 fontsize=14, fontweight='bold', color='white', y=1.01)
    plt.show()

---
## 4️⃣ Full Implementation: Manual GD vs sklearn vs Analytical Solution

Since univariate linear regression has a **single global minimum** (the loss surface is a convex paraboloid), we can:
1. **Gradient Descent** — iteratively find the minimum
2. **sklearn's LinearRegression** — uses the normal equation (closed-form)
3. **Analytical / Closed-Form Solution** — reverse-engineer by setting gradient = 0

### The Closed-Form Solution (Setting Gradient = 0)

At the minimum, all partial derivatives equal zero:

$$\frac{\partial L}{\partial w} = 0 \implies w = \frac{\sum(x_i - \bar{x})(y_i - \bar{y})}{\sum(x_i - \bar{x})^2}$$

$$\frac{\partial L}{\partial b} = 0 \implies b = \bar{y} - w\bar{x}$$

All three methods should produce **exactly the same result**!

In [ ]:
# ── 4a: Load toy dataset from sklearn ───────────────────────────────
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression

np.random.seed(42)
X_toy, y_toy = make_regression(n_samples=50, n_features=1, noise=10, random_state=42)
x_flat = X_toy.flatten()

print(f'📊 Dataset: {len(x_flat)} samples, 1 feature')
print(f'   x range: [{x_flat.min():.2f}, {x_flat.max():.2f}]')
print(f'   y range: [{y_toy.min():.2f}, {y_toy.max():.2f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_flat, y_toy, color=C['cyan'], s=50, edgecolors='white',
           linewidth=1, alpha=0.9)
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('Toy Dataset from sklearn.datasets.make_regression',
             fontweight='bold', color=C['cyan'])
plt.tight_layout()
plt.show()

In [ ]:
# ── 4b: Method 1 — Manual Gradient Descent ─────────────────────────
def gradient_descent_full(x, y, lr=0.05, n_iters=1000):
    """Manual gradient descent for y = b + wx using formulas from the notes."""
    w, b = 0.0, 0.0
    n = len(x)
    w_history, b_history = [w], [b]
    loss_history = [np.mean((b + w * x - y) ** 2)]

    for _ in range(n_iters):
        errors = (b + w * x) - y
        # ∂L/∂w = (2/n) Σ (b + w·xi - yi)·xi
        dL_dw = (2 / n) * np.sum(errors * x)
        # ∂L/∂b = (2/n) Σ (b + w·xi - yi)
        dL_db = (2 / n) * np.sum(errors)

        w -= lr * dL_dw
        b -= lr * dL_db

        loss = np.mean((b + w * x - y) ** 2)
        w_history.append(w); b_history.append(b)
        loss_history.append(loss)

    return w, b, w_history, b_history, loss_history

w_gd, b_gd, w_hist_gd, b_hist_gd, loss_hist_gd = gradient_descent_full(
    x_flat, y_toy, lr=0.05, n_iters=500
)

print('━' * 50)
print('🔧 Method 1 — Manual Gradient Descent')
print('━' * 50)
print(f'  w = {w_gd:.6f}')
print(f'  b = {b_gd:.6f}')
print(f'  Final MSE = {loss_hist_gd[-1]:.6f}')

In [ ]:
# ── 4c: Method 2 — sklearn LinearRegression ────────────────────────
model_sk = LinearRegression()
model_sk.fit(X_toy, y_toy)
w_sk = model_sk.coef_[0]
b_sk = model_sk.intercept_
mse_sk = np.mean((model_sk.predict(X_toy) - y_toy) ** 2)

print('━' * 50)
print('📦 Method 2 — sklearn LinearRegression')
print('━' * 50)
print(f'  w = {w_sk:.6f}')
print(f'  b = {b_sk:.6f}')
print(f'  MSE = {mse_sk:.6f}')

In [ ]:
# ── 4d: Method 3 — Analytical (Closed-Form) Solution ───────────────
#
# Setting the gradients to zero and solving:
#
#   ∂L/∂w = 0  ⟹  Σ (b + w·xi - yi)·xi = 0
#   ∂L/∂b = 0  ⟹  Σ (b + w·xi - yi)     = 0
#
# From ∂L/∂b = 0:
#   n·b + w·Σxi = Σyi
#   b = ȳ - w·x̄
#
# Substituting into ∂L/∂w = 0:
#   w = Σ(xi - x̄)(yi - ȳ) / Σ(xi - x̄)²

x_mean = np.mean(x_flat)
y_mean = np.mean(y_toy)

w_analytical = np.sum((x_flat - x_mean) * (y_toy - y_mean)) / np.sum((x_flat - x_mean) ** 2)
b_analytical = y_mean - w_analytical * x_mean
mse_analytical = np.mean((b_analytical + w_analytical * x_flat - y_toy) ** 2)

print('━' * 50)
print('🧮 Method 3 — Analytical (Closed-Form) Solution')
print('━' * 50)
print('  Setting ∂L/∂w = 0 and ∂L/∂b = 0, we derive:')
print('  w = Σ(xi − x̄)(yi − ȳ) / Σ(xi − x̄)²')
print('  b = ȳ − w·x̄')
print()
print(f'  x̄ = {x_mean:.6f}')
print(f'  ȳ = {y_mean:.6f}')
print(f'  w = {w_analytical:.6f}')
print(f'  b = {b_analytical:.6f}')
print(f'  MSE = {mse_analytical:.6f}')

In [ ]:
# ── 4e: Comparison — All Three Methods Side by Side ─────────────────

comparison_html = f"""
<style>
  .cmp {{ border-collapse: collapse; width: 100%; font-family: 'Segoe UI', sans-serif; }}
  .cmp th {{ background: #0f3460; color: #00d2ff; padding: 12px 16px; text-align: center;
             font-size: 14px; border-bottom: 2px solid #00d2ff; }}
  .cmp td {{ padding: 10px 16px; text-align: center; border-bottom: 1px solid #2a2a4a;
             color: #e0e0e0; font-size: 13px; }}
  .cmp tr:nth-child(even) {{ background: #1a1a2e; }}
  .cmp tr:nth-child(odd) {{ background: #16213e; }}
  .cmp tr:hover {{ background: #1e2d50; }}
  .val {{ color: #0cebeb; font-weight: bold; font-family: monospace; font-size: 14px; }}
  .method {{ color: #f9d423; font-weight: bold; }}
</style>
<table class="cmp">
  <tr><th>Method</th><th>w</th><th>b</th><th>MSE</th></tr>
  <tr>
    <td class="method">🔧 Manual Gradient Descent</td>
    <td class="val">{w_gd:.6f}</td>
    <td class="val">{b_gd:.6f}</td>
    <td class="val">{loss_hist_gd[-1]:.6f}</td>
  </tr>
  <tr>
    <td class="method">📦 sklearn LinearRegression</td>
    <td class="val">{w_sk:.6f}</td>
    <td class="val">{b_sk:.6f}</td>
    <td class="val">{mse_sk:.6f}</td>
  </tr>
  <tr>
    <td class="method">🧮 Analytical (∂L = 0)</td>
    <td class="val">{w_analytical:.6f}</td>
    <td class="val">{b_analytical:.6f}</td>
    <td class="val">{mse_analytical:.6f}</td>
  </tr>
</table>
"""
display(HTML(comparison_html))

print('\n✅ All three methods converge to the SAME w and b!')
print('   This is because the loss surface is a convex paraboloid with a single global minimum.')
print('   → Gradient descent iterates to find it')
print('   → sklearn uses the normal equation to compute it directly')
print('   → We derived it ourselves by setting ∂L/∂w = 0 and ∂L/∂b = 0')

In [ ]:
# ── 4f: Final Visualization — All Methods Overlaid ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
xl = np.linspace(x_flat.min() - 0.5, x_flat.max() + 0.5, 100)

methods = [
    ('Manual GD',   w_gd,         b_gd,         C['accent']),
    ('sklearn',     w_sk,         b_sk,         C['teal']),
    ('Analytical',  w_analytical, b_analytical, C['yellow']),
]

# ── Left: All three lines overlaid ──
ax = axes[0]
ax.scatter(x_flat, y_toy, color=C['cyan'], s=40, edgecolors='white',
           linewidth=0.8, alpha=0.8, label='Data')
for name, w, b, color in methods:
    ax.plot(xl, b + w * xl, color=color, lw=2.5, alpha=0.85,
            label=f'{name}: y={b:.2f}+{w:.2f}x')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('All Methods Overlaid (identical!)', fontweight='bold', color='white')
ax.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#444')

# ── Middle: GD convergence ──
ax2 = axes[1]
ax2.plot(loss_hist_gd, color=C['accent'], lw=2)
ax2.fill_between(range(len(loss_hist_gd)), loss_hist_gd, alpha=0.1, color=C['accent'])
ax2.axhline(mse_analytical, color=C['yellow'], ls='--', lw=1.5, alpha=0.8,
            label=f'Analytical min = {mse_analytical:.4f}')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('MSE Loss')
ax2.set_title('GD Loss → Analytical Minimum', fontweight='bold', color=C['accent'])
ax2.legend(fontsize=9, facecolor='#1a1a2e', edgecolor='#444')

# ── Right: Contour + GD path ──
ax3 = axes[2]
wr = np.linspace(w_analytical - 40, w_analytical + 40, 100)
br = np.linspace(b_analytical - 20, b_analytical + 20, 100)
Wg, Bg = np.meshgrid(wr, br)
Lg = np.zeros_like(Wg)
for i in range(len(x_flat)):
    Lg += (Bg + Wg * x_flat[i] - y_toy[i]) ** 2
Lg /= len(x_flat)

ax3.contourf(Wg, Bg, Lg, levels=40, cmap='magma', alpha=0.9)
ax3.plot(w_hist_gd, b_hist_gd, 'o-', color=C['teal'], markersize=2,
         lw=1, alpha=0.8, label='GD path')
ax3.scatter(w_analytical, b_analytical, color=C['yellow'], s=150, marker='*',
            edgecolors='white', linewidth=2, zorder=10, label='Analytical min')
ax3.scatter(w_sk, b_sk, color=C['teal'], s=80, marker='D',
            edgecolors='white', linewidth=1.5, zorder=10, label='sklearn')
ax3.set_xlabel('w'); ax3.set_ylabel('b')
ax3.set_title('GD Path → Minimum', fontweight='bold', color=C['cyan'])
ax3.legend(fontsize=8, facecolor='#1a1a2e', edgecolor='#444')

fig.suptitle('All Three Methods Converge to the Same Solution',
             fontsize=15, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.show()

---
## 📝 Key Takeaways

| Concept | What We Learned |
|---------|----------------|
| **Linear Model** | $y = b + wx$ — just two parameters control the line |
| **Loss (MSE)** | $L = \frac{1}{I}\sum(b + wx_i - y_i)^2$ — quantifies prediction error |
| **Gradient Descent** | Iteratively updates $w$ and $b$ by following the negative gradient |
| **Convex Loss** | For linear regression, there's always a **single global minimum** |
| **Analytical Solution** | Setting $\nabla L = 0$ gives the exact optimal parameters |
| **All methods agree** | GD, sklearn, and the analytical formula all find the same answer |